In [1]:
from aggregator import Aggregator
from client import Client
from manager import Manager
import numpy as np
import os
import torch

/home/aladin/resilient_sfl/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Hyperparameters
num_rounds = 2
num_epochs = 2
num_clients = 2

# Instantiate manager
manager = Manager(name="bert-manager", model_type="bert-base-uncased", batch_size=16)

# Load data
train_dataloader, valid_dataloader, test_dataloader = manager.preprocess_dataset(glue_dataset="sst2", truncate=100)
train_split = manager.split_data(train_dataloader, clients=num_clients)
valid_split = manager.split_data(valid_dataloader, clients=num_clients)
test_split = manager.split_data(test_dataloader, clients=num_clients)

# Load model
num_labels = len(set(train_dataloader.dataset.tensors[2].tolist()))
pretrained_model = manager.load_model(num_labels=num_labels)

Found cached dataset glue (/home/aladin/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)
100%|██████████| 3/3 [00:00<00:00, 169.11it/s]
Some weights of the model checkpoint at bert-base-uncased were not used when initializing CustomBertModel: ['cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing CustomBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CustomBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassi

In [ ]:
# Global training
global_loss = []
global_accuracy = []
dir = os.getcwd()

for r in range(num_rounds):

    print(f"GLOBAL ROUND : {r+1} of {num_rounds}")

    # Instantiate clients
    for i in range(num_clients):
        client = Client(name=f"client_{i}",
                        model=pretrained_model,
                        manager=manager,
                        train_data=train_split[i],
                        valid_data=valid_split[i])
        
        if r != 0:  
            # Load model from path
            print("Loading Model from Directory...")
            checkpoint_path = dir + f"/checkpoints/{client.name}_model.pt"
            checkpoint = torch.load(checkpoint_path)
            client.model.load_state_dict(checkpoint)
        else:
            pass

        # Local training
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        client.train_model(device=device, num_epochs=num_epochs, checkpoint_path=dir+"/checkpoints")
    
    # Load client models from path
    clients = []
    for i in range(num_clients):
        client = Client(name=f"client_{i}",
                        model=pretrained_model,
                        manager=manager,
                        train_data=train_split[i],
                        valid_data=valid_split[i])

        checkpoint_path = dir + f"/checkpoints/{client.name}_model.pt"
        checkpoint = torch.load(checkpoint_path)
        client.model.load_state_dict(checkpoint)

        clients.append(client)

    # Parameter aggregation
    aggregator = Aggregator(name="bert_aggregator")

    attentions = aggregator.accumulate_attentions([client.model for client in clients])
    heads = aggregator.accumulate_heads([client.model for client in clients])
    embeddings = aggregator.accumulate_embeddings([client.model for client in clients])

    aggregated_attentions = aggregator.aggregate(attentions)
    aggregated_heads = aggregator.aggregate(heads)
    aggregated_embeddings = aggregator.aggregate(embeddings)

    # Model update and save
    for client in clients:
        client.update_model(aggregated_attentions)
        client.update_model(aggregated_heads)
        client.update_model(aggregated_embeddings)

        model_path = dir + f"/checkpoints/{client.name}_model.pt"
        client.save_model(path=model_path)

    # Track global model performance
    global_model = clients[-1]
    loss, acc = global_model.evaluate_global_model(device=device, valid_data=valid_dataloader) 
    global_loss.append(loss)
    global_accuracy.append(acc)
    

# Save final model
final_model = clients[-1].model
final_model_path = dir + "/checkpoints/fedavg_bert_sst2_model.pt"
torch.save(final_model.state_dict(), final_model_path)

In [ ]:
# Plot global model performance
manager.save_plots(loss=global_loss, 
                   accuracy=global_accuracy, 
                   title="FedAvg BERT Training",
                   path=dir + "/plots/fedavg_bert_sst2.png")

In [ ]:
# Save global model metrics
metric_path = dir + "/metrics/"
manager.save_metric(metric=global_loss, path=metric_path+"global_loss")
manager.save_metric(metric=global_accuracy, path=metric_path+"global_accuracy")

In [ ]:
# Retrieve model metrics
loss_metric_path = metric_path + "global_loss.npy"
accuracy_metric_path = metric_path + "global_accuracy.npy"

global_loss = np.load(loss_metric_path)
global_accuracy = np.load(accuracy_metric_path)

print(global_loss)
print("--" * 10)
print(global_accuracy)

In [ ]:
# Evaluate client from directory
client = Client(name="eval_client",
                model=pretrained_model,
                manager=manager,
                train_data=train_split[0],
                valid_data=valid_split[0])

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_path = dir + "/checkpoints/fedavg_bert_sst2_model.pt"
client.evaluate_model(model_path=model_path, device=device, valid_data=valid_dataloader)